# Diabetes Scenario 2 — Who comes back? Predictors of readmission within 30 days

_Building a small prognostic model on electronic health record data. Level: second analysis. About two hours._

**Before you start.** These are 101,766 inpatient encounters of 71,518 patients with a diabetes
diagnosis in 130 US hospitals, 1999 to 2008, from an electronic health record warehouse: nobody
was randomised, HbA1c was rarely measured, several drug classes of today did not exist, and a
readmission to a hospital outside the network is invisible. One row is **one encounter**; a
patient with several stays appears several times, so the analyses keep the first encounter of
each patient and drop the encounters that ended in death or hospice, as the published analysis
did. Codes: 0 = No, 1 = Yes; Gender 1 = Male, 2 = Female; age only as 10-year bands.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
encounter record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

**Learning goals.** Compare continuous measures between readmitted and not readmitted patients
with a t-test; test categorical factors with chi-square; recognise a dose-response pattern in a
count variable; fit a logistic regression with a handful of covariates and read adjusted odds
ratios; and understand why the strongest predictor of future use of hospital care is past use.

**Clinical background.** Readmission within 30 days of discharge is a widely used quality
indicator; in the United States, hospitals with high rates are financially penalised. Hospitals
therefore try to identify patients at high risk before discharge. Candidate predictors are
available in any record system: how the patient came in, how long they stayed, how many
diagnoses and drugs they had, where they were discharged to, and above all how often they had
been in hospital before. A prognostic model tells you who is likely to be readmitted, not why,
and not what would prevent it: discharge to a nursing facility predicts readmission because it
marks a frail patient, not because nursing facilities cause readmissions.

## 1. Connect to the MIP and open the Diabetes pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("Diabetes")    # add version="DiabetesUCI" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ds = dm.datasets.list()[0]              # one dataset: Diabetes 130-US hospitals 1999-2008
display(dm)
display(ds)

In [ ]:
# The variable tree follows the encounter: Encounter, Admission, Demographics, Diagnoses
# (primary, secondary, summary), Hospital care, Prior utilisation, Laboratory, Diabetes
# medication (individual drugs, summary), Discharge, Outcome.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the Diabetes 130 dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ds], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",)):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    table.attrs["outcome_levels"] = list(lab_y.values())
    table.attrs["events"] = [lab_y.get(e, e) for e in events]
    return table


def collapse(table, groups):
    """Pool the rows of a crosstab into two groups ({name: [row labels]}) and test the 2x2 table
    with a chi-square with Yates' correction, as the platform does, from the aggregate counts."""
    first = table.columns[0]
    levels, events = table.attrs["outcome_levels"], table.attrs["events"]
    rows = []
    for name, members in groups.items():
        part = table[table[first].isin(members)]
        counts = {lev: int(part[lev].sum()) for lev in levels}
        n = sum(counts.values())
        hit = sum(counts[e] for e in events)
        rows.append({first: name, "n": n, **counts, "% " + "/".join(events): 100 * hit / n if n else np.nan})
    out = pd.DataFrame(rows)
    (a, b), (c, d) = [(r["n"] - sum(r[e] for e in events), sum(r[e] for e in events)) for r in rows[:2]]
    n = a + b + c + d
    chi2 = n * (abs(a * d - b * c) - n / 2) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
    out.attrs["chi2"], out.attrs["p"] = chi2, math.erfc(math.sqrt(chi2 / 2))
    return out


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of encounters with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under Diabetes) | Role | Values |
|---|---|---|---|---|
| `first_encounter` | First encounter of the patient | Encounter | cohort filter: one encounter per patient; curation-derived | 0 No; 1 Yes |
| `death_or_hospice` | Died or discharged to hospice | Discharge | cohort filter: encounters that could be readmitted; curation-derived | 0 No; 1 Yes |
| `readmit_30` | Readmitted within 30 days | Outcome | outcome; curation-derived | 0 No; 1 Yes |
| `number_inpatient` | Inpatient visits, previous year | Prior utilisation | predictor, count | integer, 0-30 |
| `number_emergency` | Emergency visits, previous year | Prior utilisation | predictor, count | integer, 0-100 |
| `time_in_hospital` | Length of stay | Encounter | predictor | integer, 1-14 days |
| `number_diagnoses` | Number of diagnoses | Diagnoses / Summary | predictor | integer, 1-16 |
| `num_medications` | Number of medications | Hospital care | predictor | integer, 0-100 |
| `age_mid` | Age, band midpoint | Demographics | predictor, continuous; curation-derived | integer, 5-95 years |
| `age_group3` | Age group (3 levels) | Demographics | predictor, grouped; curation-derived | 1 Under 30 years; 2 30-59 years; 3 60 years and over |
| `discharge_home` | Discharged to home | Discharge | predictor; curation-derived | 0 No; 1 Yes |
| `admission_source_grp` | Admission source group | Admission | predictor; curation-derived | 1 Emergency room; 2 Physician, clinic or HMO referral; 3 Other or not recorded |
| `diag_1_grp` | Primary diagnosis group | Diagnoses / Primary diagnosis | predictor; curation-derived | 9 codes, listed in the tree |
| `gender` | Gender | Demographics | predictor | 1 Male; 2 Female; 9 Unknown/Invalid |
| `hba1c_group` | HbA1c group (published analysis) | Laboratory | covariate of Step 5; curation-derived | 1 Not measured; 2 Measured, result below 8%; 3 Measured, above 8%, diabetes medication changed; 4 Measured, above 8%, diabetes medication not changed |
| `discharge_disposition_id` | Discharge disposition | Discharge | discussion question 2 | 26 codes, listed in the tree |
| `n_encounters` | Number of encounters of this patient | Encounter | extension; curation-derived | integer, 1-40 |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `A1Cresult`) is what
the csv column is called; you will see it in some result payloads.

In [ ]:
readmit    = dm.variables["Readmitted within 30 days"]
inpatient  = dm.variables["Inpatient visits, previous year"]
emergency  = dm.variables["Emergency visits, previous year"]
stay       = dm.variables["Length of stay"]
diagnoses  = dm.variables["Number of diagnoses"]
meds       = dm.variables["Number of medications"]
age_mid    = dm.variables["Age, band midpoint"]
age3       = dm.variables["Age group (3 levels)"]
home       = dm.variables["Discharged to home"]
source     = dm.variables["Admission source group"]
diag       = dm.variables["Primary diagnosis group"]
gender     = dm.variables["Gender"]
hba1c_grp  = dm.variables["HbA1c group (published analysis)"]
display(mip.to_frame([readmit, inpatient, emergency, stay, diagnoses, meds, age_mid, age3, home, source, diag, gender, hba1c_grp]))

## Step 1. Continuous predictors, one at a time

The cohort of Scenario 1 (first encounter, no death or hospice: 69,973 encounters). An
*Independent t-test* of each of six measures between readmitted and not readmitted encounters,
with the two group means from one `describe()` per group.

*Expected means, readmitted versus not:* stay 4.8 versus 4.2 days; prior inpatient visits 0.37
versus 0.16; prior emergency visits 0.15 versus 0.10; medications 16.6 versus 15.6; diagnoses 7.5
versus 7.2; age 67.8 versus 65.2 years. All p < 0.001. Readmitted patients were sicker on every
count, but the differences are modest except for prior admissions, which are more than doubled.

In [ ]:
first = dm.variables["First encounter of the patient"]
death = dm.variables["Died or discharged to hospice"]
cohort = (F(first) == "1") & (F(death) == "0")      # one encounter per patient, able to be readmitted

measures = [stay, inpatient, emergency, meds, diagnoses, age_mid]
means = {}
for label, c in (("readmitted", "1"), ("not readmitted", "0")):
    res = analysis(measures, cohort & (F(readmit) == c)).describe()
    means[label] = numeric_summary(res, measures).set_index("variable")["mean"]
rows = []
for v in measures:
    t = analysis([v, readmit], cohort, drop_missing=True).t_test(variable=v, group_by=readmit, group_a="1", group_b="0")
    s = ttest_summary(t)
    rows.append({"measure": v.label, "mean, readmitted": means["readmitted"][v.label],
                 "mean, not readmitted": means["not readmitted"][v.label], "difference": s["mean_diff"],
                 "t": s["t_stat"], "p": s["p"], "n": s["n_obs"]})
display(pd.DataFrame(rows).style.format({"mean, readmitted": "{:.2f}", "mean, not readmitted": "{:.2f}",
                                         "difference": "{:.2f}", "t": PCT, "p": fmt_p}).hide(axis="index"))

## Step 2. Prior admissions as a dose-response

The number of inpatient visits in the previous year is a count: one `describe()` of the outcome
per band (0, 1, 2, 3 or more), through filters.

*Expected:*

| Prior inpatient visits | n | Readmitted within 30 days |
|---|---|---|
| 0 | 61,782 | 8.1% |
| 1 | 5,794 | 12.9% |
| 2 | 1,501 | 18.5% |
| 3 or more | 896 | 26.5% |

Each additional admission in the previous year raises the risk by about 5 percentage points.
This is the single most useful predictor in the dataset.

In [ ]:
bands = {"0": F(inpatient) == 0, "1": F(inpatient) == 1, "2": F(inpatient) == 2, "3 or more": F(inpatient) >= 3}
by_prior = band_rates([readmit], bands, base_filter=cohort)
display(by_prior.style.format({readmit.label: PCT}).hide(axis="index"))
ax = by_prior.set_index("band")[readmit.label].plot(kind="bar", color="#4C72B0", figsize=(5, 3), rot=0)
ax.set_ylabel("% readmitted within 30 days"); ax.set_xlabel("inpatient visits in the previous year")
ax.set_title("Past use predicts future use")
plt.tight_layout()
plt.show()

## Step 3. Categorical predictors

Readmission cross-tabulated against five categorical factors. The unknown gender (3 encounters)
is under the platform's privacy threshold and does not appear in the table, although the test
counts it.

*Expected percentages readmitted:*

| Factor | Levels | p |
|---|---|---|
| Discharged home | yes 6.9%, no 12.5% | < 0.001 |
| Age group | under 30: 6.2%, 30 to 59: 7.2%, 60 and over: 9.9% | < 0.001 |
| Admission source | emergency 9.2%, referral 8.6%, other 8.7% | 0.02 |
| Primary diagnosis | injury 10.8%, circulatory 9.7%, diabetes 9.1%, respiratory 7.3%, others 8.0 to 9.1% | < 0.001 |
| Gender | men 8.9%, women 9.0% | 0.74 |

Gender predicts nothing; the paper dropped it for the same reason. Discharge destination is the
strongest categorical factor.

In [ ]:
factors = []
for exposure in (home, age3, source, diag, gender):
    t = crosstab(exposure, readmit, base_filter=cohort, event_label="readmitted")
    print(f"{exposure.label}: chi2 = {t.attrs['chi2']:.1f}, p = {fmt_p(t.attrs['p'])}")
    for _, r in t.iterrows():
        factors.append({"factor": exposure.label, "level": r[exposure.label], "n": r["n"], "% readmitted": r["% readmitted"]})
factors = pd.DataFrame(factors)
display(factors.style.format({"% readmitted": PCT}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 6))
labels = factors["factor"] + ": " + factors["level"].astype(str).str.slice(0, 40)
ax.barh(labels[::-1], factors["% readmitted"][::-1], color="#4C72B0")
ax.set_xlabel("% readmitted within 30 days"); ax.set_title("Readmission by each factor")
plt.tight_layout()
plt.show()

## Step 4. A small logistic regression

A *Logistic Regression* of readmission on the number of prior inpatient visits, the length of
stay, the age group and the discharge destination. The platform compares each category with its
first code: under 30 years for the age group, and not discharged home for the discharge. The
paper's parametrisation (reference 30 to 59, and home) follows by dividing: 60 and over versus
30 to 59 is 1.35 / 1.09 = 1.24, under 30 versus 30 to 59 is 1 / 1.09 = 0.92, and not home versus
home is 1 / 0.58 = 1.72.

*Expected (69,973 encounters), as the platform shows it:*

| Term | Odds ratio | 95% CI |
|---|---|---|
| Prior inpatient visits, per visit | 1.43 | 1.39 to 1.48 |
| Length of stay, per day | 1.03 | 1.02 to 1.04 |
| Age 30 to 59 (vs under 30) | 1.09 | 0.89 to 1.34 |
| Age 60 and over (vs under 30) | 1.35 | 1.11 to 1.65 |
| Discharged to home = Yes (vs not home) | 0.58 | 0.55 to 0.62 |

Each prior admission multiplies the odds of early readmission by 1.4, independently of age, stay
and discharge destination. Under 30 is not different from 30 to 59 once the others are in the
model: the raw difference in Step 3 was small and that group is small (1,808).

In [ ]:
covariates = [inpatient, stay, age3, home]
model = analysis([readmit] + covariates, cohort, drop_missing=True).logistic_regression(
    x=covariates, y=readmit, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} encounters in the model")
ors = odds_ratio_table(model, covariates).set_index("term")
show_odds_ratios(ors.reset_index(), "Odds of readmission within 30 days, first encounters")

# The paper's parametrisation, from the platform's terms
a30, a60, home_yes = ors.loc["Age group (3 levels) = 30-59 years"], ors.loc["Age group (3 levels) = 60 years and over"], ors.loc["Discharged to home = Yes"]
print(f"60 and over versus 30 to 59: {a60['odds_ratio'] / a30['odds_ratio']:.2f}; "
      f"under 30 versus 30 to 59: {1 / a30['odds_ratio']:.2f}; "
      f"not discharged home versus home: {1 / home_yes['odds_ratio']:.2f} "
      f"(95% CI {1 / home_yes['ci_high']:.2f} to {1 / home_yes['ci_low']:.2f})")

## Step 5. Does HbA1c add anything?

The same model with the paper's four HbA1c groups added (reference: not measured, the first code).

*Expected:* odds ratios 0.96, 1.01 and 0.91 for the three measured groups, none significant (p
from 0.2 to 0.9). Once prior admissions and discharge destination are accounted for, measuring
HbA1c is no longer associated with readmission in the whole cohort. This is consistent with
Scenario 1, where the association existed only in diabetes admissions. It shows why the paper
needed interaction terms, and why the conclusion of an analysis depends on what is in the model.

In [ ]:
covariates = [inpatient, stay, age3, home, hba1c_grp]
model = analysis([readmit] + covariates, cohort, drop_missing=True).logistic_regression(
    x=covariates, y=readmit, positive_class="1")
ors = odds_ratio_table(model, covariates)
ors["term"] = ors["term"].str.replace("HbA1c group (published analysis) = Measured, ", "HbA1c measured, ", regex=False)
show_odds_ratios(ors, "The same model with the HbA1c groups")

## Step 6. State the result

A model answer:

> Among 69,973 first hospital encounters of patients with diabetes, 9.0% were readmitted within
> 30 days. The strongest predictor was the number of inpatient admissions in the previous year:
> 8% with none, 13% with one, 19% with two, 27% with three or more, an adjusted odds ratio of 1.4
> per admission. Not being discharged home (odds ratio 1.7), age 60 and over (1.2) and a longer
> stay (1.03 per day) added information; gender and the admission source did not. Such a model
> identifies high-risk patients for discharge planning but does not tell which interventions
> would keep them out of hospital.

**Questions for discussion.** The odds ratio for the length of stay is 1.03 per day, highly
significant: is it clinically important? Compare a 2-day and a 10-day stay. Why is "not
discharged home" a predictor, and would an intervention on the discharge destination change
readmission? Prior admissions predict future admissions: what could a hospital do with that? The
model uses first encounters only: what problem does applying it to every discharge raise?

**Optional extension.** Fit the model of Step 4 on all encounters that could be readmitted and
compare the odds ratios; then think about why the number of encounters of the patient should not
be used as a predictor.

In [ ]:
could_be_readmitted = F(death) == "0"
covariates = [inpatient, stay, age3, home]
model = analysis([readmit] + covariates, could_be_readmitted, drop_missing=True).logistic_regression(
    x=covariates, y=readmit, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} encounters in the model")
display(odds_ratio_table(model, covariates).style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": fmt_p}).hide(axis="index"))